In [16]:
library(readxl)
library(tidyverse)
library(igraph)
library(writexl)

In [23]:
path_in  <- "C:\\Users\\cfutr\\OneDrive\\Escritorio\\CICS\\2025\\Tesis\\Objetivos de Aprendizaje\\Modelo\\datos\\Raw_Data\\Matriz_de_Adyacencia_Pensiones_v6.xlsx"
path_out <- "C:\\Users\\cfutr\\OneDrive\\Escritorio\\CICS\\2025\\Tesis\\Objetivos de Aprendizaje\\Modelo\\datos\\Intermedias\\A_min_con_w_option1.xlsx"

df <- read_excel(path_in, sheet = "Hoja2",col_names = TRUE)

A <- df %>%
  rename(node = 1) %>%                            # primera columna = nombres fila
  mutate(across(-node, ~replace_na(as.numeric(.x), 0))) %>%  # NA->0, a numérico
  column_to_rownames("node") %>%
  as.matrix()

stopifnot(nrow(A) == ncol(A))
stopifnot(all(A %in% c(0, 1)))
stopifnot(identical(rownames(A), colnames(A)))

nodes <- rownames(A)
id_map <- tibble(id = seq_along(nodes), concept = nodes)
A[1:5, 1:5]

New names:
• `` -> `...1`


,Que_es_pension,Pension_Autofinanciada,AFP,Compania_Seguros,Saldo_Cuenta_Individual
Que_es_pension,0,1,0,0,0
Pension_Autofinanciada,0,0,1,1,0
AFP,0,0,0,0,1
Compania_Seguros,0,0,0,0,0
Saldo_Cuenta_Individual,0,0,0,0,0


In [24]:
## Crear grafo dirigido a partir de la matriz de adyacencia (sin bucles)
A_tmp <- A
diag(A_tmp) <- 0

g_min <- graph_from_adjacency_matrix(A_tmp, mode = "directed", diag = FALSE)

is_dag(g_min)
c(vcount(g_min), ecount(g_min))

[1] TRUE

[1]  36 151

In [25]:
# Calcular métricas de importancia estructural para cada nodo

nodes <- V(g_min)$name

# Descendientes (alcanzabilidad)
D <- distances(g_min, v = nodes, to = nodes, mode = "out")
reach <- is.finite(D) & D > 0
desc <- rowSums(reach)

# Out-degree y betweenness
outdeg <- degree(g_min, mode = "out")
btw <- betweenness(g_min, directed = TRUE, normalized = TRUE)

# Normalizar a [0,1]
norm01 <- function(x) (x - min(x)) / (max(x) - min(x) + 1e-9)

w_tbl <- id_map %>%
  mutate(
    descendants  = desc[concept],
    outdeg       = outdeg[concept],
    betweenness  = btw[concept],
    descendants_n = norm01(descendants),
    outdeg_n      = norm01(outdeg),
    btw_n         = norm01(betweenness)
  )

In [26]:
alpha <- 0.6
beta  <- 0.15
delta <- 0.25

w_tbl_fix <- w_tbl %>%
  mutate(
    w_raw = alpha*descendants_n + beta*outdeg_n + delta*btw_n
  )

pos <- w_tbl_fix$w_raw[w_tbl_fix$w_raw > 1e-12]
eps <- 0.05 * as.numeric(quantile(pos, 0.10))  # 5% del p10

w_tbl_fix <- w_tbl_fix %>%
  mutate(
    w_raw2 = ifelse(w_raw <= 1e-12, eps, w_raw),
    w = w_raw2 / sum(w_raw2),
    rank = rank(-w, ties.method = "min")
  )

w_tbl_fix %>%
  arrange(rank) %>%
  select(rank, id, concept, w, descendants, outdeg, betweenness) %>%
  head(15)

rank,id,concept,w,descendants,outdeg,betweenness
<int>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,15,Modalidades_Pensiones,0.07411564,31,9,0.034251701
2,24,Retiro_Programado,0.06640856,26,11,0.030466186
3,30,Beneficiarios_Legales,0.06577463,23,12,0.035464186
4,3,AFP,0.06302284,29,9,0.020112045
5,19,Renta_Vitalicia,0.06266720,24,12,0.027478992
6,1,Que_es_pension,0.06115587,33,13,0.000000000
7,5,Saldo_Cuenta_Individual,0.05482607,28,9,0.008431373
8,16,Riesgo de longevidad,0.05402815,32,7,0.000000000
9,2,Pension_Autofinanciada,0.05353228,32,5,0.002410964


In [27]:
write_xlsx(
  list(
    A_min = df,
    w_option1 = w_tbl_fix %>% arrange(id)  # mantiene el orden de A_min
  ),
  path = path_out
)

Estos son códigos para generar distintas combinaciones de pesos, en general los resultados arrojan que no son tan diferentes unas de otras (más menos la misma importancia de los conceptos independiente del peso asignado a cada criterio)

In [ ]:
alpha <- 0.25; beta <- 0.25; delta <- 0.5

w_tbl2 <- w_tbl %>%
  mutate(
    w_raw = alpha*descendants_n + beta*outdeg_n + delta*btw_n,
    w = w_raw/sum(w_raw),
    rank = rank(-w, ties.method="min")
  ) %>%
  arrange(rank)

w_tbl2 %>% select(rank, concept, w, descendants, outdeg, betweenness) %>% head(12)
# sources (indegree 0) en g_min
sources <- V(g_min)$name[degree(g_min, mode="in") == 0]
sources
w_tbl2 %>% filter(concept %in% sources) %>% arrange(rank)

In [ ]:
set.seed(1)

# Baseline (ajusta si tu baseline fue otro)
base_alpha <- 0.25   # descendants
base_beta  <- 0.25  # outdeg
base_gamma <- 0.5  # betweenness

# Ranking baseline
baseline <- w_tbl %>%
  mutate(
    w_base = base_alpha*descendants_n + base_beta*outdeg_n + base_gamma*btw_n,
    w_base = w_base / sum(w_base),
    rank_base = rank(-w_base, ties.method = "min")
  ) %>%
  select(concept, w_base, rank_base)

# ---- 1) Generar 20 combinaciones de pesos que suman 1 ----
# Método: muestreo Dirichlet "casero" (gamma + normalizar)
n_scen <- 20
W <- matrix(rexp(n_scen*3, rate = 1), ncol = 3)
W <- W / rowSums(W)
colnames(W) <- c("alpha_desc", "beta_outdeg", "gamma_btw")

weights_df <- as_tibble(W) %>%
  mutate(scenario = row_number())

weights_df

In [ ]:
# ---- 2) Recalcular rankings por escenario ----
scenarios <- weights_df %>%
  tidyr::crossing(w_tbl %>% select(concept, descendants_n, outdeg_n, btw_n)) %>%
  mutate(
    w_raw = alpha_desc*descendants_n + beta_outdeg*outdeg_n + gamma_btw*btw_n
  ) %>%
  group_by(scenario) %>%
  mutate(
    w = w_raw / sum(w_raw),
    rank = rank(-w, ties.method = "min")
  ) %>%
  ungroup()

# ---- 3) Spearman vs baseline ----
cmp <- scenarios %>%
  left_join(baseline, by = "concept") %>%
  group_by(scenario, alpha_desc, beta_outdeg, gamma_btw) %>%
  summarise(
    spearman = cor(rank, rank_base, method = "spearman"),
    .groups = "drop"
  ) %>%
  arrange(desc(spearman))

cmp
# Top escenarios más parecidos al baseline
cmp %>% slice(1:5)

In [ ]:
# ---- 4) Estabilidad Top-10: % de escenarios donde cada concepto aparece en Top-10 ----
topk <- 10
stability <- scenarios %>%
  mutate(in_top10 = rank <= topk) %>%
  group_by(concept) %>%
  summarise(
    p_top10 = mean(in_top10),
    avg_rank = mean(rank),
    sd_rank = sd(rank),
    .groups = "drop"
  ) %>%
  arrange(desc(p_top10), avg_rank)

stability %>% slice(1:15)

In [ ]:
# ---- 5) (Opcional) ver el Top-10 de un escenario particular ----
# elige uno (por ejemplo el más distinto al baseline)
worst_s <- cmp %>% slice_tail(n=1) %>% pull(scenario)

scenarios %>%
  filter(scenario == worst_s) %>%
  arrange(rank) %>%
  select(concept, rank, w) %>%
  slice(1:10)